# Hypothesis 2 Testing

_If someone reads an increasing number of articles over a time period, they are more likely to subscribe to newsletters._

## Caveats

TKTK

## Clarification

- "articles" -> only look at page views of articles, i.e., `PAGE_IS_ARTICLE is True`.
- "increasing number of articles" -> we're interested in trajectory of count for each user/reader
- "more likely to subscribe to newsletters" -> we can keep 0 or 1 as the boolean response

Since calculating the correlation between the trajectory of a variable and a binary outcome is not as straightforward as doing so between said variable and said outcome, for this hypothesis, before diving into correlations, let's do a visual analysis first.

One way of doing this is to create two subpopulations of readers: subscribers and non-subscribers. For each subpopulation, we can plot out the average trajectory of article count over time (after rebasing a user's events' timestamps to the timestamp of their first event). We'll also use a rolling count as opposed to a raw count to make it easier to detect trends from trajectories.

If our hypothesis holds true, we should see an upward average trajectory among subscribers. The same plot for non-subscribers serves as a point of reference.

## Preprocessing

In [ ]:
from enum import auto

import pandas as pd
from helpers.pathlib import get_path_site_checkpoint
from leaspy import AlgorithmSettings as LeaspyAlgorithmSettings
from leaspy import Data as LeaspyData
from leaspy import Leaspy

from ata_pipeline1.helpers.enums import FieldNew, FieldSnowplow, StrEnum
from ata_pipeline1.helpers.urllib import append_slash
from ata_pipeline1.site import (
    AFRO_LA,
    DALLAS_FREE_PRESS,
    OPEN_VALLEJO,
    THE_19TH,
    SiteName,
)

In [ ]:
# Nov. 3, 2022 to Jan. 25, 2022 (inclusively) a.k.a 12 weeks
CSV_PREFIX = "221103-230125"

In [ ]:
# INSTRUCTION: If there's a new field you'd like to add, add its name to this enum
class FieldTemp(StrEnum):
    ARTICLE_COUNT_PERIOD_INDEX = auto()
    ARTICLE_COUNT_PERIOD_START_DATE = auto()
    #     ARTICLE_COUNT_WINDOW_END_DATE = auto()
    #     DAYS_FROM_EARLIEST_TO_SUBMISSION = auto()
    DERIVED_TSTAMP_DATE = auto()
    EARLIEST_EVENT_TSTAMP = auto()
    EARLIEST_EVENT_DATE = auto()
    FIRST_PERIOD_TSTAMP = auto()
    LAST_EVENT_TSTAMP = auto()
    LAST_EVENT_DATE = auto()
    LAST_SUBMISSION_TSTAMP = auto()
    LAST_SUBMISSION_DATE = auto()
    LAST_SUBMISSION_PERIOD_INDEX = auto()
    NUM_PERIODS = auto()
    NUM_ARTICLES_READ = auto()
    SUBMITTED = auto()

In [ ]:
# Read data from CHECKPOINT 7
df_afla = pd.read_pickle(get_path_site_checkpoint(CSV_PREFIX, 7, AFRO_LA.name))
df_dfp = pd.read_pickle(get_path_site_checkpoint(CSV_PREFIX, 7, DALLAS_FREE_PRESS.name))
df_ov = pd.read_pickle(get_path_site_checkpoint(CSV_PREFIX, 7, OPEN_VALLEJO.name))
df_19 = pd.read_pickle(get_path_site_checkpoint(CSV_PREFIX, 7, THE_19TH.name))

In [ ]:
# Filter to page views of articles and page views where a newsletter-form submission was recorded
def get_articles_and_submissions_dfs(df, site_name):
    df_articles = df.query(f"{FieldNew.PAGE_IS_ARTICLE} " + f"and {FieldNew.DWELL_SECS} >= 10 ")
    df_submissions = df.query(FieldNew.FORM_SUBMIT_IS_NEWSLETTER)
    print(f"Site: {site_name}")
    print(f"Found {df_articles.shape[0]} article views that last more than 10 seconds")
    print(f"Found {df_submissions.shape[0]} page views where a newsletter form is submitted")
    print("\n")
    return df_articles, df_submissions


df_afla_articles, df_afla_submissions = get_articles_and_submissions_dfs(df_afla, AFRO_LA.name)
df_dfp_articles, df_dfp_submissions = get_articles_and_submissions_dfs(df_dfp, DALLAS_FREE_PRESS.name)
df_ov_articles, df_ov_submissions = get_articles_and_submissions_dfs(df_ov, OPEN_VALLEJO.name)
df_19_articles, df_19_submissions = get_articles_and_submissions_dfs(df_19, THE_19TH.name)

### Reformat page URL paths

In [ ]:
# Make article URLs of uniform format so that nunique() later returns accurate results
def make_uniform_urlpaths(df_articles):
    df_articles = df_articles.copy()
    df_articles[FieldSnowplow.PAGE_URLPATH] = df_articles[FieldSnowplow.PAGE_URLPATH].apply(append_slash)
    return df_articles


df_afla_articles = make_uniform_urlpaths(df_afla_articles)
df_dfp_articles = make_uniform_urlpaths(df_dfp_articles)
df_ov_articles = make_uniform_urlpaths(df_ov_articles)
df_19_articles = make_uniform_urlpaths(df_19_articles)

### Specify unit and length of rolling count window

In [ ]:
# INSTRUCTIONS: Specify window length as a number of time periods
ROLLING_WINDOW_PERIODS = 3
# INSTRUCTIONS: Specify length of time period as a number of days. This is the unit of the rolling window.
# At the pre-rolling level, article views should be aggregated into single-period counts
PERIOD_LENGTH_DAYS = 1

### Get all unique users

In [ ]:
# Group events by user and get the date of earliest event for each user
def group_events(df, site_name):
    df_grouped = df.groupby(FieldSnowplow.DOMAIN_USERID).aggregate(
        **{
            FieldTemp.EARLIEST_EVENT_TSTAMP: pd.NamedAgg(column=FieldSnowplow.DERIVED_TSTAMP, aggfunc="min"),
            FieldTemp.LAST_EVENT_TSTAMP: pd.NamedAgg(column=FieldSnowplow.DERIVED_TSTAMP, aggfunc="max"),
        }
    )
    df_grouped[FieldTemp.EARLIEST_EVENT_DATE] = df_grouped[FieldTemp.EARLIEST_EVENT_TSTAMP].dt.date
    df_grouped[FieldTemp.LAST_EVENT_DATE] = df_grouped[FieldTemp.LAST_EVENT_TSTAMP].dt.date
    # Add a column showing user lifetime in terms of periods (e.g., if window unit is 2 day,
    # and a reader's lifetime is 4 days, that's 2 units, but if the lifetime is 3 days, that's just 1 period)
    df_grouped[FieldTemp.NUM_PERIODS] = (
        df_grouped[FieldTemp.LAST_EVENT_DATE] - df_grouped[FieldTemp.EARLIEST_EVENT_DATE]
    ).dt.days // PERIOD_LENGTH_DAYS
    print(f"Site: {site_name}")
    print(f"Found {df_grouped.shape[0]} unique readers")
    print(f"Minimum no. periods: {df_grouped[FieldTemp.NUM_PERIODS].min()}")
    print(f"Median no. periods: {df_grouped[FieldTemp.NUM_PERIODS].median()}")
    print(f"Maximum no. periods: {df_grouped[FieldTemp.NUM_PERIODS].max()}")
    print("\n")
    return df_grouped


df_afla_users = group_events(df_afla, AFRO_LA.name)
df_dfp_users = group_events(df_dfp, DALLAS_FREE_PRESS.name)
df_ov_users = group_events(df_ov, OPEN_VALLEJO.name)
df_19_users = group_events(df_19, THE_19TH.name)

### Get all unique users who sign up (a.k.a. subscribers)

In [ ]:
# Group events by user. If a user has more than one event, take the most recent one to ensure X is
# as big as it can be
def group_submissions(df_submissions, df_users, site_name):
    df_grouped = (
        df_submissions.groupby(FieldSnowplow.DOMAIN_USERID)
        .aggregate(
            **{FieldTemp.LAST_SUBMISSION_TSTAMP: pd.NamedAgg(column=FieldSnowplow.DERIVED_TSTAMP, aggfunc="max")}
        )
        .merge(df_users[FieldTemp.EARLIEST_EVENT_DATE], left_index=True, right_index=True)
    )
    df_grouped[FieldTemp.LAST_SUBMISSION_DATE] = df_grouped[FieldTemp.LAST_SUBMISSION_TSTAMP].dt.date
    df_grouped[FieldTemp.SUBMITTED] = True
    df_grouped[FieldTemp.LAST_SUBMISSION_PERIOD_INDEX] = (
        df_grouped[FieldTemp.LAST_SUBMISSION_DATE] - df_grouped[FieldTemp.EARLIEST_EVENT_DATE]
    ).dt.days // PERIOD_LENGTH_DAYS
    print(f"Site: {site_name}")
    print(f"Found {df_grouped.shape[0]} users who signed up for newsletter")
    print(
        f"Minimum period index where a submission happens: {df_grouped[FieldTemp.LAST_SUBMISSION_PERIOD_INDEX].min()}"
    )
    print(f"Median period index a submission happens: {df_grouped[FieldTemp.LAST_SUBMISSION_PERIOD_INDEX].median()}")
    print(f"Maximum period index a submission happens: {df_grouped[FieldTemp.LAST_SUBMISSION_PERIOD_INDEX].max()}")
    print("\n")
    return df_grouped


df_afla_users_submission = group_submissions(df_afla_submissions, df_afla_users, AFRO_LA.name)
df_dfp_users_submission = group_submissions(df_dfp_submissions, df_dfp_users, DALLAS_FREE_PRESS.name)
df_ov_users_submission = group_submissions(df_ov_submissions, df_ov_users, OPEN_VALLEJO.name)
df_19_users_submission = group_submissions(df_19_submissions, df_19_users, THE_19TH.name)

### Remove unqualified users

In [ ]:
# Get all users who have been on a site for less than TWO consecutive rolling windows,
# since at least two are required to determine trajectory/trend
def get_unqualified_users(df_users, site_name: SiteName):
    all_users = df_users.index
    assert all_users.is_unique

    threshold_periods = ROLLING_WINDOW_PERIODS + 1
    df_users_unqualified = df_users.query(f"{FieldTemp.NUM_PERIODS} < {threshold_periods}")

    unqualified_users = df_users_unqualified.index
    num_users = len(all_users)
    num_unqualified_users = len(unqualified_users)
    num_qualified_users = num_users - num_unqualified_users
    print(f"Site: {site_name}")
    print(
        f"Found {num_qualified_users} ({num_qualified_users / num_users:.1%}) users "
        + f"whose lifetime on the site lasts more than {threshold_periods} periods, "
        + f"1 period equal {PERIOD_LENGTH_DAYS} day(s)"
    )
    print("\n")
    return unqualified_users


users_unqualified_afla = get_unqualified_users(df_afla_users, AFRO_LA.name)
users_unqualified_dfp = get_unqualified_users(df_dfp_users, DALLAS_FREE_PRESS.name)
users_unqualified_ov = get_unqualified_users(df_ov_users, OPEN_VALLEJO.name)
users_unqualified_19 = get_unqualified_users(df_19_users, THE_19TH.name)

In [ ]:
# Get all subscribing users who have been on a site for less than THREE consecutive rolling windows,
# since at least three are required to determine trajectory/trend in the days BEFORE the sign-up date
# (which should at least be two days)
def get_unqualified_subscribers(df_users_submission, site_name: SiteName):
    all_users = df_users_submission.index
    assert all_users.is_unique

    threshold_periods = ROLLING_WINDOW_PERIODS + 2
    df_users_unqualified = df_users_submission.query(f"{FieldTemp.LAST_SUBMISSION_PERIOD_INDEX} < {threshold_periods}")

    unqualified_users = df_users_unqualified.index
    num_users = len(all_users)
    num_unqualified_users = len(unqualified_users)
    num_qualified_users = num_users - num_unqualified_users
    print(f"Site: {site_name}")
    print(
        f"Found {num_qualified_users} ({num_qualified_users / num_users:.1%}) subscribing users "
        + f"whose age on site on last newsletter subscription date is older than {threshold_periods} periods, "
        + f"1 period equal {PERIOD_LENGTH_DAYS} day(s)"
    )
    print("\n")
    return unqualified_users


subscribers_unqualified_afla = get_unqualified_subscribers(df_afla_users_submission, AFRO_LA.name)
subscribers_unqualified_dfp = get_unqualified_subscribers(df_dfp_users_submission, DALLAS_FREE_PRESS.name)
subscribers_unqualified_ov = get_unqualified_subscribers(df_ov_users_submission, OPEN_VALLEJO.name)
subscribers_unqualified_19 = get_unqualified_subscribers(df_19_users_submission, THE_19TH.name)

In [ ]:
# Get all unqualified users
users_unqualified_afla = users_unqualified_afla.union(subscribers_unqualified_afla)
users_unqualified_dfp = users_unqualified_dfp.union(subscribers_unqualified_dfp)
users_unqualified_ov = users_unqualified_ov.union(subscribers_unqualified_ov)
users_unqualified_19 = users_unqualified_19.union(subscribers_unqualified_19)

In [ ]:
# Remove unqualified users from user, subscriber and article-event DataFrames
def remove_unqualified_users_as_index(df, users_unqualified, site_name: SiteName, label_print="users"):
    df = df.drop(labels=users_unqualified)
    num_users_unqualified = len(users_unqualified)
    num_users = num_users_unqualified + df.shape[0]
    print(f"Site: {site_name}")
    print(
        f"Removed {num_users_unqualified} ({num_users_unqualified / num_users:.1%}) unqualified {label_print} "
        + f"out of all {num_users} {label_print}"
    )
    print("\n")
    return df


def remove_unqualified_users_as_data(df, users_unqualified):
    return df[~df[FieldSnowplow.DOMAIN_USERID].isin(users_unqualified)]


# Users
df_afla_users = remove_unqualified_users_as_index(df_afla_users, users_unqualified_afla, AFRO_LA.name)
df_dfp_users = remove_unqualified_users_as_index(df_dfp_users, users_unqualified_dfp, DALLAS_FREE_PRESS.name)
df_ov_users = remove_unqualified_users_as_index(df_ov_users, users_unqualified_ov, OPEN_VALLEJO.name)
df_19_users = remove_unqualified_users_as_index(df_19_users, users_unqualified_19, THE_19TH.name)

# Users who subscribe a.k.a. subscribers
df_afla_users_submission = remove_unqualified_users_as_index(
    df_afla_users_submission, subscribers_unqualified_afla, AFRO_LA.name, "subscribers"
)
df_dfp_users_submission = remove_unqualified_users_as_index(
    df_dfp_users_submission, subscribers_unqualified_dfp, DALLAS_FREE_PRESS.name, "subscribers"
)
df_ov_users_submission = remove_unqualified_users_as_index(
    df_ov_users_submission, subscribers_unqualified_ov, OPEN_VALLEJO.name, "subscribers"
)
df_19_users_submission = remove_unqualified_users_as_index(
    df_19_users_submission, subscribers_unqualified_19, THE_19TH.name, "subscribers"
)

# Article events
df_afla_articles = remove_unqualified_users_as_data(df_afla_articles, users_unqualified_afla)
df_dfp_articles = remove_unqualified_users_as_data(df_dfp_articles, users_unqualified_dfp)
df_ov_articles = remove_unqualified_users_as_data(df_ov_articles, users_unqualified_ov)
df_19_articles = remove_unqualified_users_as_data(df_19_articles, users_unqualified_19)

### Group article views by time period

In [ ]:
def group_article_events(df_articles, df_users, include_zero_count_periods: bool = True):
    def create_period_indices(row):
        derived_tstamp_date = row[FieldSnowplow.DERIVED_TSTAMP].date()
        domain_userid = row[FieldSnowplow.DOMAIN_USERID]
        user_first_event_date = df_users.loc[domain_userid][FieldTemp.EARLIEST_EVENT_DATE]
        row[FieldTemp.ARTICLE_COUNT_PERIOD_INDEX] = (
            derived_tstamp_date - user_first_event_date
        ).days // PERIOD_LENGTH_DAYS
        return row

    df_grouped = (
        df_articles.copy()
        .apply(create_period_indices, axis=1)
        .groupby([FieldSnowplow.DOMAIN_USERID, FieldTemp.ARTICLE_COUNT_PERIOD_INDEX])[FieldSnowplow.PAGE_URLPATH]
        .nunique()
        .rename(FieldTemp.NUM_ARTICLES_READ)
    )

    df_grouped = pd.DataFrame(df_grouped)

    if include_zero_count_periods:
        all_periods = df_users[FieldTemp.NUM_PERIODS].map(lambda n: range(0, n, 1)).explode()
        df_all_periods = (
            pd.DataFrame({FieldTemp.ARTICLE_COUNT_PERIOD_INDEX: all_periods, FieldTemp.NUM_ARTICLES_READ: 0})
            .reset_index()
            .set_index([FieldSnowplow.DOMAIN_USERID, FieldTemp.ARTICLE_COUNT_PERIOD_INDEX])
        )
        df_grouped = (df_all_periods + df_grouped).fillna(0).astype(int)

    return df_grouped.reset_index(FieldTemp.ARTICLE_COUNT_PERIOD_INDEX)

In [ ]:
df_afla_users_article = group_article_events(df_afla_articles, df_afla_users)
df_dfp_users_article = group_article_events(df_dfp_articles, df_dfp_users)
df_ov_users_article = group_article_events(df_ov_articles, df_ov_users)
# df_19_users_article = group_article_events(df_19_articles, df_19_users)

### Split users into subscribers and non-subscribers

In [ ]:
# ARTICLE COUNTS BY TIME PERIOD OF SUBSCRIBERS
def get_article_counts_for_subscribers(df_users_article, df_users_submission):
    return df_users_article.merge(df_users_submission, how="inner", left_index=True, right_index=True).query(
        f"{FieldTemp.ARTICLE_COUNT_PERIOD_INDEX} < {FieldTemp.LAST_SUBMISSION_PERIOD_INDEX}"
    )[[FieldTemp.ARTICLE_COUNT_PERIOD_INDEX, FieldTemp.NUM_ARTICLES_READ]]


df_afla_subscribers_article_count = get_article_counts_for_subscribers(df_afla_users_article, df_afla_users_submission)
df_dfp_subscribers_article_count = get_article_counts_for_subscribers(df_dfp_users_article, df_dfp_users_submission)
df_ov_subscribers_article_count = get_article_counts_for_subscribers(df_ov_users_article, df_ov_users_submission)
# df_19_subscribers_article_count = get_article_counts_for_subscribers(df_19_users_article, df_19_users_submission)

In [ ]:
# ARTICLE COUNTS BY TIME PERIOD OF NON-SUBSCRIBERS
def get_article_counts_for_nonsubscribers(df_users_article, df_users_submission):
    return df_users_article.merge(df_users_submission, how="left", left_index=True, right_index=True).query(
        f"{FieldTemp.SUBMITTED} != True"
    )[[FieldTemp.ARTICLE_COUNT_PERIOD_INDEX, FieldTemp.NUM_ARTICLES_READ]]


df_afla_nonsubscribers_article_count = get_article_counts_for_nonsubscribers(
    df_afla_users_article, df_afla_users_submission
)
df_dfp_nonsubscribers_article_count = get_article_counts_for_nonsubscribers(
    df_dfp_users_article, df_dfp_users_submission
)
df_ov_nonsubscribers_article_count = get_article_counts_for_nonsubscribers(df_ov_users_article, df_ov_users_submission)
# df_19_nonsubscribers_article_count = get_article_counts_for_nonsubscribers(df_19_users_article, df_19_users_submission)

In [ ]:
def test_sub_nonsub_integrity(df_subscribers_article_count, df_nonsubscribers_article_count, df_users_article):
    subs = df_subscribers_article_count.index
    nonsubs = df_nonsubscribers_article_count.index
    #     users = df_users_article.index
    # Assert everyone is either subscriber or nonsubscriber
    assert len(subs.intersection(nonsubs)) == 0


test_sub_nonsub_integrity(df_afla_subscribers_article_count, df_afla_nonsubscribers_article_count, df_afla_users)
test_sub_nonsub_integrity(df_dfp_subscribers_article_count, df_dfp_nonsubscribers_article_count, df_dfp_users)
test_sub_nonsub_integrity(df_ov_subscribers_article_count, df_ov_nonsubscribers_article_count, df_ov_users)
# test_sub_nonsub_integrity(df_19_subscribers_article_count, df_19_nonsubscribers_article_count, df_19_users)

### Split users into subscribers and non-subscribers

We'll use the Python library [Leaspy](https://leaspy.readthedocs.io/en/stable/index.html) to calculate the average article-count trajectory for each of the subpupulations. Leaspy has specific requirements in terms of input data format, which we need to follow.

In [ ]:
class FieldLeaspy(StrEnum):
    ID = "ID"
    TIME = "TIME"
    NUM_ARTICLES_READ = "NUM_ARTICLES_READ"

In [ ]:
def transform_leaspy_format(df_article_count):
    return (
        df_article_count.reset_index()
        .rename(
            columns={
                FieldSnowplow.DOMAIN_USERID: FieldLeaspy.ID,
                FieldTemp.ARTICLE_COUNT_PERIOD_INDEX: FieldLeaspy.TIME,
                FieldTemp.NUM_ARTICLES_READ: FieldLeaspy.NUM_ARTICLES_READ,
            }
        )
        .set_index([FieldLeaspy.ID, FieldLeaspy.TIME])
        .sort_index()
    )


# Transform subscriber counts
df_afla_final_sub = transform_leaspy_format(df_afla_subscribers_article_count)
df_dfp_final_sub = transform_leaspy_format(df_dfp_subscribers_article_count)
df_ov_final_sub = transform_leaspy_format(df_ov_subscribers_article_count)
# df_19_final_sub = transform_leaspy_format(df_19_subscribers_article_count)

# Transform non-subscriber counts
df_afla_final_nonsub = transform_leaspy_format(df_afla_nonsubscribers_article_count)
df_dfp_final_nonsub = transform_leaspy_format(df_dfp_nonsubscribers_article_count)
df_ov_final_nonsub = transform_leaspy_format(df_ov_nonsubscribers_article_count)
# df_19_final_nonsub = transform_leaspy_format(df_19_nonsubscribers_article_count)

## Analysis

In [ ]:
RANDOM_SEED = 42

### AfroLA

In [ ]:
df_afla_final_sub.shape

The DataFrame holding article counts of AfroLA newsletter subscribers before they subscribe is empty. This is expected since in the time frame we're working it, AfroLA is still pre-launch. We'll need to look at the DataFrame holding article counts of non-subscribers.

In [ ]:
def fit(df_final, random_seed=RANDOM_SEED, n_iter=1000):
    data = LeaspyData.from_dataframe(df_final)
    model = Leaspy("univariate_linear")
    settings = LeaspyAlgorithmSettings("mcmc_saem", seed=RANDOM_SEED, n_iter=n_iter)
    model.fit(data, settings)
    return model

In [ ]:
model_afla_nonsub = fit(df_afla_final_nonsub, n_iter=8000)

In [ ]:
df_afla_final_nonsub.loc["04c43396-84bd-4ea1-9e0b-c6945455245b"]